# UNet layer activation diagnostics

This notebook drives the current `imp evaluate --save-layer` interface, loads captured intermediate activations, and plots them as tiled feature-map grids. The research question is:

> **Does the UNet use ERA5 information when trained with the naive linear encoders?**

The flow is:

1. Point the notebook at a compatible real-data checkpoint and configuration.
2. Run `imp evaluate` with one `--save-layer` option per module to hook. `ActivationSaver` records the first activation for each selected layer in every test batch, together with the raw batch tensors.
3. Load the per-batch `.pt` files and compare the encoder and processor feature maps.

Only the evaluation/test path is instrumented. The implementation is `icenet_mp.callbacks.ActivationSaver`.


## UNet pipeline shapes (`quick_test`, `start_out_channels=64`)

The tracked `sample` configuration uses the current `sample_south` multimodal data configuration and the `quick_test` `EncodeProcessDecode` model. ARGO, OSISAF and ERA5 inputs are all represented on the 432×432 southern EASE2 grid before their naive encoders rescale them to the 128×128 latent grid.

The exact input-channel count depends on the variables present in each configured dataset, so the table keeps those channel dimensions symbolic rather than hard-coding an old dataset layout.

| Stage | Hook path (`processor.*` unless noted) | Saved tensor shape | Notes |
| --- | --- | --- | --- |
| ARGO encoder output | `encoder_float_argo` | `(N, C_argo, 128, 128)` | One encoded slice from the ARGO input group |
| OSISAF encoder output | `encoder_sic_osisaf` | `(N, C_sic, 128, 128)` | One encoded slice from the sea-ice input group |
| ERA5 encoder output | `encoder_era5` | `(N, C_era5, 128, 128)` | One encoded slice from the atmospheric input group |
| Combined history passed to processor | *(no module; processor input)* | `(N, 3 × ΣC_source, 128, 128)` | Three history steps are folded into channels for the current 2-day forecast configuration |
| `conv1` block | `conv1` | `(N, 64, 128, 128)` | First UNet block |
| after `maxpool1` | `maxpool1` | `(N, 64, 64, 64)` | Stride-2 downsample |
| `conv2` block | `conv2` | `(N, 128, 64, 64)` | 64 → 128 channels |
| after `maxpool2` | `maxpool2` | `(N, 128, 32, 32)` | |
| `conv3` block | `conv3` | `(N, 256, 32, 32)` | 128 → 256 channels |
| after `maxpool3` | `maxpool3` | `(N, 256, 16, 16)` | |
| `conv4` block | `conv4` | `(N, 256, 16, 16)` | 256 → 256 channels |
| after `maxpool4` | `maxpool4` | `(N, 256, 8, 8)` | |
| `conv5` bottleneck | `conv5` | `(N, 512, 8, 8)` | 256 → 512 channels |
| `up6b` block | `up6b` | `(N, 256, 16, 16)` | Upsample + skip connection |
| `up7b` block | `up7b` | `(N, 256, 32, 32)` | Upsample + skip connection |
| `up8b` block | `up8b` | `(N, 128, 64, 64)` | Upsample + skip connection |
| `up9b` block | `up9b` | `(N, 64, 128, 128)` | Upsample + skip connection |
| final projection | `final_layer` | `(N, C_latent_out, 128, 128)` | Per-pixel channel projection |

Module names above match the current `UNetProcessor` and the encoder names registered from the current data-group names.


## 1. Parameters

This diagnostic needs a checkpoint trained with the configuration and datasets you want to investigate. It deliberately does not ship with a machine-specific checkpoint path.

Before starting Jupyter, set the checkpoint path. The tracked `sample` configuration is the default; override it only when the checkpoint was trained with another root config.

```bash
export ICENET_MP_DIAGNOSTICS_CHECKPOINT=/path/to/checkpoint.ckpt
# Optional when needed:
# export ICENET_MP_DIAGNOSTICS_CONFIG=sample
# export ICENET_MP_DIAGNOSTICS_BASE_PATH=/path/to/base
```

The evaluation itself uses the local-file logger so it does not require a W&B login. The required real datasets must already exist under the selected base path; creating ERA5 data still requires the normal CDS credentials.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CONFIG_NAME = os.environ.get("ICENET_MP_DIAGNOSTICS_CONFIG", "sample")
BASE_PATH = Path(
    os.environ.get("ICENET_MP_DIAGNOSTICS_BASE_PATH", REPO_ROOT / "../base")
).expanduser().resolve()

checkpoint_value = os.environ.get("ICENET_MP_DIAGNOSTICS_CHECKPOINT")
if not checkpoint_value:
    raise RuntimeError(
        "Set ICENET_MP_DIAGNOSTICS_CHECKPOINT to a compatible checkpoint before running this notebook."
    )
CHECKPOINT = Path(checkpoint_value).expanduser().resolve()
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")

ACTIVATIONS_DIR = BASE_PATH / "evaluation" / "activations"
LAYER_PATHS = [
    "encoder_float_argo",
    "encoder_sic_osisaf",
    "encoder_era5",
    "processor.conv1",
    "processor.conv5",
    "processor.final_layer",
]
HYDRA_OVERRIDES = [f'base_path="{BASE_PATH}"', "loggers=local_files"]

print("REPO_ROOT       :", REPO_ROOT)
print("CONFIG_NAME     :", CONFIG_NAME)
print("BASE_PATH       :", BASE_PATH)
print("CHECKPOINT      :", CHECKPOINT)
print("ACTIVATIONS_DIR :", ACTIVATIONS_DIR)
print("LAYER_PATHS     :")
for layer_path in LAYER_PATHS:
    print("  -", layer_path)


## 2. Run `imp evaluate` with activation capture

The cell below invokes the same CLI used elsewhere in IceNet-MP. `--save-layer` is repeated once for every requested module path.

Equivalent terminal form:

```bash
imp evaluate \
  --config-name sample \
  --checkpoint /path/to/checkpoint.ckpt \
  --save-layer encoder_era5 \
  --save-layer processor.conv1 \
  loggers=local_files
```

`ActivationSaver` writes `batch_00000.pt`, `batch_00001.pt`, ... plus `metadata.json` under `${base_path}/evaluation/activations/`.


In [ ]:
import shlex
import subprocess

cmd: list[str] = [
    "imp",
    "evaluate",
    "--config-name",
    CONFIG_NAME,
    "--checkpoint",
    str(CHECKPOINT),
]
for layer_path in LAYER_PATHS:
    cmd += ["--save-layer", layer_path]
cmd += HYDRA_OVERRIDES

ACTIVATIONS_DIR.mkdir(parents=True, exist_ok=True)
for old_batch in ACTIVATIONS_DIR.glob("batch_*.pt"):
    old_batch.unlink()
metadata_path = ACTIVATIONS_DIR / "metadata.json"
if metadata_path.exists():
    metadata_path.unlink()

print("$", shlex.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## 3. Load captured activations

Each evaluation batch produces one `batch_{idx:05d}.pt` file containing:

- `batch_idx`;
- `layer_paths`;
- `activations`, a mapping from requested module path to the captured NCHW tensor;
- `inputs`, the raw tensor entries copied from the test batch when input saving is enabled.


In [ ]:
import json
import torch

metadata_path = ACTIVATIONS_DIR / "metadata.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Activation metadata not found: {metadata_path}")
metadata = json.loads(metadata_path.read_text())
print("metadata:", json.dumps(metadata, indent=2))

batch_files = sorted(ACTIVATIONS_DIR.glob("batch_*.pt"))
if not batch_files:
    raise FileNotFoundError(f"No activation batches found in {ACTIVATIONS_DIR}")
print(f"
{len(batch_files)} batch file(s) in {ACTIVATIONS_DIR}")
for path in batch_files[:5]:
    print(f"  - {path.name}")

payload = torch.load(batch_files[0], map_location="cpu", weights_only=False)

print(f"
batch_idx: {payload['batch_idx']}")
print("
activations:")
for name, tensor in payload["activations"].items():
    print(f"  {name:30s} {tuple(tensor.shape)}  {tensor.dtype}")

print("
inputs (raw batch tensors):")
for name, tensor in payload.get("inputs", {}).items():
    print(f"  {name:30s} {tuple(tensor.shape)}  {tensor.dtype}")


## 4. Visualise each layer in turn

For each hooked layer, plot a tiled grid of channel feature maps from the first batch element (`N=0`). When a layer has many channels (e.g. `conv5` has 512), only the first `MAX_CHANNELS_PER_LAYER` are shown.

In [ ]:
import matplotlib.pyplot as plt

BATCH_ELEMENT = 0
MAX_CHANNELS_PER_LAYER = 16
TILE_COLS = 4


def plot_feature_maps(title: str, nchw: torch.Tensor) -> None:
    chw = nchw[BATCH_ELEMENT].float().cpu()
    n_channels = min(MAX_CHANNELS_PER_LAYER, chw.shape[0])
    cols = min(TILE_COLS, n_channels)
    rows = (n_channels + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 2.5 * rows))
    flat_axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for idx in range(n_channels):
        flat_axes[idx].imshow(chw[idx].numpy(), cmap="viridis")
        flat_axes[idx].set_title(f"ch {idx}", fontsize=9)
        flat_axes[idx].axis("off")
    for idx in range(n_channels, len(flat_axes)):
        flat_axes[idx].axis("off")
    channel_note = (
        f"all {chw.shape[0]}"
        if n_channels == chw.shape[0]
        else f"first {n_channels} of {chw.shape[0]}"
    )
    fig.suptitle(
        f"{title}\nshape NCHW {tuple(nchw.shape)} — showing {channel_note} channels",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

payload = torch.load(batch_files[BATCH_ELEMENT], map_location="cpu", weights_only=False)
for name, tensor in payload["activations"].items():
    plot_feature_maps(name, tensor)


## 5. Reading the plots

- **`encoder_era5`** shows the encoded atmospheric inputs; comparing it with the sea-ice and ARGO encoders helps separate source-specific structure before fusion.
- **`processor.conv1`** shows the first learned representation after the encoded history streams are combined.
- **`processor.conv5`** is the 8×8 bottleneck. Varied, spatially structured channels suggest the model is carrying distinct large-scale regimes through the compressed representation.
- Comparing batches with similar sea-ice state but different atmospheric conditions is more informative than interpreting one activation map in isolation.

To inspect another batch, change the selected `batch_files` index in the load/plot cells and rerun them. For a different experiment, use a checkpoint and root config that match each other rather than overriding the data definition independently at evaluation time.
